In [2]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [3]:
class PICOExtraction(BaseModel):
    population: str = Field(description="The characteristics of the patients, target population, or problem.")
    intervention: str = Field(description="The main treatment, procedure, or intervention being studied.")
    comparison: Optional[str] = Field(description="The alternative or control treatment. Return null if none exists.")
    outcome: str = Field(description="The primary results or effects measured in the study.")

In [4]:
llm = OllamaLLM(model="llama3.1", format="json", temperature=0)

In [5]:
parser = JsonOutputParser(pydantic_object=PICOExtraction)

In [6]:
template = """
You are an expert medical researcher. Read the following medical abstract and extract the PICO elements, along with their word positioning.

{format_instructions}

Medical Abstract:
{abstract}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["abstract"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

In [7]:
pico_pipeline = prompt | llm | parser

if __name__ == "__main__":
    sample_abstract = """
    This study evaluated the efficacy of a new mRNA vaccine compared to the standard viral vector vaccine 
    in preventing severe respiratory distress in adults over 65 with pre-existing cardiac conditions. 
    Over a 12-month period, the mRNA group showed a 40% reduction in hospitalizations, with no significant 
    increase in adverse cardiovascular events.
    """
    
    print("Extracting PICO...")
    result = pico_pipeline.invoke({"abstract": sample_abstract})
    
    import json
    print(json.dumps(result, indent=2))

Extracting PICO...
{
  "population": "adults over 65 with pre-existing cardiac conditions",
  "intervention": "new mRNA vaccine",
  "comparison": "standard viral vector vaccine",
  "outcome": "preventing severe respiratory distress"
}


# IOU EVALUATION

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import Optional
from pydantic import BaseModel, Field
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from difflib import SequenceMatcher

class PICOExtraction(BaseModel):
    population: Optional[str] = Field(description="The patients or problem. Return null if none.")
    intervention: Optional[str] = Field(description="The main treatment. Return null if none.")
    comparison: Optional[str] = Field(description="The control. Return null if none.")
    outcome: Optional[str] = Field(description="The primary results. Return null if none.")

llm = OllamaLLM(model="llama3.1", format="json", temperature=0)
parser = JsonOutputParser(pydantic_object=PICOExtraction)

def extract_text_from_mask(text, mask):
    """
    Converts a binary mask (1s and 0s) back into readable text strings 
    to use as ground-truth examples in the prompt.
    """
    words = text.split()
    extracted_spans = []
    current_span = []
    
    for word, label in zip(words, mask):
        if label == 1:
            current_span.append(word)
        else:
            if current_span:
                extracted_spans.append(" ".join(current_span))
                current_span = []
                
    if current_span: # catch any trailing span
        extracted_spans.append(" ".join(current_span))
        
    if not extracted_spans:
        return "null"
        
    return " ; ".join(extracted_spans)


def evaluate_extraction(original_text, extracted_text, gt_mask, threshold=0.6):
    """Calculates precision against binary masks (0s and 1s)."""
    if not extracted_text or str(extracted_text).lower() in ('null', 'none'):
        return "None", None 
        
    orig_words = original_text.split()
    ext_words = extracted_text.split()
    
    if not orig_words or not ext_words:
        return "None", None
        
    window = len(ext_words)
    best_ratio, best_start = 0, 0
    
    for i in range(len(orig_words) - window + 1):
        window_text = " ".join(orig_words[i:i+window])
        ratio = SequenceMatcher(None, extracted_text.lower(), window_text.lower()).ratio()
        if ratio > best_ratio:
            best_ratio, best_start = ratio, i

    if best_ratio < threshold:
        return "Hallucinated", 0.0
        
    mask_slice = gt_mask[best_start : best_start + window]
    
    if any(label == 1 for label in mask_slice):
        return "Hit (Relaxed)", 1.0
        
    return "Miss", 0.0

def main():
    data = np.load('./data/ebm_nlp_2_00/processed/ebm_abstracts_full.npz', allow_pickle=True)
    
    train_texts = data['train_texts']
    train_masks = {'Pop': data['train_p'], 'Int': data['train_i'], 'Out': data['train_o']}
    
    test_texts = data['test_texts']
    test_masks = {'Pop': data['test_p'], 'Int': data['test_i'], 'Out': data['test_o']}

    num_shots = 1
    few_shot_string = ""
    
    for i in range(num_shots):
        text = train_texts[i]
        pop_str = extract_text_from_mask(text, train_masks['Pop'][i])
        int_str = extract_text_from_mask(text, train_masks['Int'][i])
        out_str = extract_text_from_mask(text, train_masks['Out'][i])
        
        few_shot_string += f"--- EXAMPLE {i+1} ---\n"
        few_shot_string += f"Abstract: {text}\n"
        few_shot_string += f"Output: {{\"population\": \"{pop_str}\", \"intervention\": \"{int_str}\", \"comparison\": \"null\", \"outcome\": \"{out_str}\"}}\n\n"

    template = """You are a medical researcher. Extract PICO elements in JSON format.
    CRITICAL RULE: Your extractions MUST be exact, continuous substrings copied directly from the abstract. Do not add or change any words.
    
    {format_instructions}
    
    {few_shot_examples}
    
    --- ACTUAL TASK ---
    Abstract: {abstract}
    Output:"""

    pipeline = PromptTemplate(
        template=template,
        input_variables=["abstract"],
        partial_variables={
            "format_instructions": parser.get_format_instructions(),
            "few_shot_examples": few_shot_string.strip() # Injecting our generated shots
        }
    ) | llm | parser

    limit = len(test_texts)
    results = []
    print(f"Starting Relaxed Extraction & Label Verification on {limit} samples...")

    for i in tqdm(range(limit), desc="Processing Abstracts"):
        text = test_texts[i]
        try:
            res = pipeline.invoke({"abstract": text})
            row = {"Text": text}
            
            for short_key, full_key in [('Pop', 'population'), ('Int', 'intervention'), ('Out', 'outcome')]:
                extracted = res.get(full_key)
                status, prec = evaluate_extraction(text, extracted, test_masks[short_key][i])
                
                row.update({
                    f"{short_key}_Extracted": extracted,
                    f"{short_key}_Status": status,
                    f"{short_key}_Precision": prec
                })
                
            results.append(row)
                
        except Exception as e:
            print(f"\nError processing sample {i}: {e}") 

    if results:
        df = pd.DataFrame(results)
        df.to_csv("pico_1_shot_results.csv", index=False)
        
        print("\n" + "="*50 + "\nTRUE RELAXED PRECISION SCORES\n" + "="*50)
        for key in ['Pop', 'Int', 'Out']:
            valid_mask = df[f'{key}_Status'].isin(['Hit (Relaxed)', 'Miss'])
            mean_prec = df.loc[valid_mask, f'{key}_Precision'].mean()
            hallucinations = (df[f'{key}_Status'] == 'Hallucinated').sum()
            
            prec_str = f"{mean_prec * 100:.1f}%" if pd.notna(mean_prec) else "N/A"
            print(f"{key:<12} Precision : {prec_str:<6} | Hallucinations: {hallucinations}")
            
        print("="*50)

if __name__ == "__main__":
    main()

Starting Relaxed Extraction & Label Verification on 184 samples...


Processing Abstracts:   5%|█                   | 10/184 [01:15<21:11,  7.31s/it]

In [2]:
df = pd.read_csv("pico_3_shot_results.csv")
        
print("\n" + "="*50 + "\nTRUE RELAXED PRECISION SCORES\n" + "="*50)
for key in ['Pop', 'Int', 'Out']:
    valid_mask = df[f'{key}_Status'].isin(['Hit (Relaxed)', 'Miss'])
    mean_prec = df.loc[valid_mask, f'{key}_Precision'].mean()
    hallucinations = (df[f'{key}_Status'] == 'Hallucinated').sum()
            
    prec_str = f"{mean_prec * 100:.1f}%" if pd.notna(mean_prec) else "N/A"
    print(f"{key:<12} Precision : {prec_str:<6} | Hallucinations: {hallucinations}")
            
print("="*50)


TRUE RELAXED PRECISION SCORES
Pop          Precision : 93.4%  | Hallucinations: 1
Int          Precision : 81.7%  | Hallucinations: 53
Out          Precision : 80.4%  | Hallucinations: 72


In [3]:
df = pd.read_csv("pico_2_shot_results.csv")
        
print("\n" + "="*50 + "\nTRUE RELAXED PRECISION SCORES\n" + "="*50)
for key in ['Pop', 'Int', 'Out']:
    valid_mask = df[f'{key}_Status'].isin(['Hit (Relaxed)', 'Miss'])
    mean_prec = df.loc[valid_mask, f'{key}_Precision'].mean()
    hallucinations = (df[f'{key}_Status'] == 'Hallucinated').sum()
            
    prec_str = f"{mean_prec * 100:.1f}%" if pd.notna(mean_prec) else "N/A"
    print(f"{key:<12} Precision : {prec_str:<6} | Hallucinations: {hallucinations}")
            
print("="*50)


TRUE RELAXED PRECISION SCORES
Pop          Precision : 92.9%  | Hallucinations: 1
Int          Precision : 89.3%  | Hallucinations: 5
Out          Precision : 70.1%  | Hallucinations: 20
